In [1]:
from langchain.agents import create_agent, AgentState
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()
model = ChatOllama(model='qwen2.5:3b')

In [2]:
from langgraph.checkpoint.memory import InMemorySaver
class CustomStateAgent(AgentState):
    user_id:str
    preferences:dict
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    state_schema=CustomStateAgent
)
result = agent.invoke({
    'messages':[{'role':'user', 'content':'Hello'}],
    'user_id':'user123',
    'preferences':{'theme':'dark'}
    },
    {'configurable':{'thread_id':'1'}}
    )
result2 = agent.invoke({
    'messages':[{'role':'user','content':'Who are you?'}]},
    {'configurable':{'thread_id':'2'}}
    )
final_result=agent.invoke({
    'messages':[{'role':'user','content':'What is the latest date you remember?'}]},
    {'configurable':{'thread_id':'1'}}
    )
final_result

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='014aa924-24c1-4398-b86c-db098e90489a'),
  AIMessage(content="Hello! How can I assist you today? Whether it's providing information on a specific topic, helping with a task, or just chatting, feel free to let me know how I can help you.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-25T11:08:18.778443Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12900647900, 'load_duration': 5273971000, 'prompt_eval_count': 30, 'prompt_eval_duration': 2292365000, 'eval_count': 41, 'eval_duration': 5228696700, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bf4d6-f137-7322-96d9-551f4707d124-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 41, 'total_tokens': 71}),
  HumanMessage(content='What is the latest date you remember?', additional_kwargs={}, response_metadata={}, id='408a9b0c-9b1c-43c

In [3]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any

@before_model
def trim_messgs(state:AgentState, runtime:Runtime)->dict[str,Any]|None:
    """Keep only last few relevant messages to fit the context window of the model."""
    messages = state['messages']
    if len(messages)<=3:
        return None
    first_message = messages[0]
    recent_messages = messages[-3:] if len(messages)%2==0 else messages[-4:]
    new_message = [first_message]+recent_messages
    return {
        'messages':[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_message
        ]
    }
agent = create_agent(
    model=model,
    middleware=[trim_messgs],
    checkpointer=InMemorySaver()
)
config:RunnableConfig = {'configurable':{'thread_id':'1'}}
agent.invoke({'messages':'Hi I am Chandler Bing'}, config)
agent.invoke({'messages':'What date is it?'}, config)
agent.invoke({'messages':'What is your job?'}, config)
final_msg = agent.invoke({'messages':'What is my name?'}, config)
final_msg['messages'][-1].pretty_print()

================================== Ai Message ==================================

I understand that you're asking "What is my name?" but I need more context or information about who "my" refers to. Are you asking for your own name, someone else's name, or do you mean something else entirely? Could you please provide a bit more detail so I can assist you better?


In [4]:
from langchain.agents.middleware import after_model
@after_model
def delete_messgs(state:AgentState, runtime:Runtime):
    """Delete all messages except first-one from the context."""
    messages = state['messages'] 
    first_msg = messages[0]
    last_msg = messages[-1]
    msg_contxt = [first_msg] + [last_msg]
    if len(messages) > 2:
        return {
            'messages' : [RemoveMessage(id=REMOVE_ALL_MESSAGES), *msg_contxt],
        }
agent = create_agent(
    model=model,
    middleware=[delete_messgs],
    checkpointer=InMemorySaver()
)
config:RunnableConfig = {'configurable':{'thread_id':'2'}}
agent.invoke({'messages':'Hi I am Joey Tribbiani'},config)
agent.invoke({'messages':'What date is it?'}, config)
agent.invoke({'messages':'What is your job?'}, config)
final_resp = agent.invoke({'messages':'What is my name?'},config)
final_resp

{'messages': [HumanMessage(content='Hi I am Joey Tribbiani', additional_kwargs={}, response_metadata={}, id='f1c74027-fabd-4958-abda-b9140f16a5ad'),
  AIMessage(content='Your name is Joey Tribbiani. Is there anything specific you would like to know or discuss regarding this name?', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-25T11:09:48.6767804Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7499453200, 'load_duration': 224254300, 'prompt_eval_count': 111, 'prompt_eval_duration': 4562513700, 'eval_count': 24, 'eval_duration': 2664577100, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bf4d8-65b7-7331-a3e6-1d9d4d542aca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 24, 'total_tokens': 135})]}

In [5]:
from langchain.agents.middleware import SummarizationMiddleware
agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=('tokens', 200),
            keep=('messages', 4)
        )
    ],
    checkpointer=InMemorySaver(),
)
config:RunnableConfig = {'configurable':{'thread_id':'1'}}
agent.invoke({'messages':'Hi I am Joey Tribbiani'}, config)
agent.invoke({'messages':'What date is today?'}, config)
agent.invoke({'messages':'Who are you?'},config)
final_res = agent.invoke({'messages':'Who am I?'}, config)
final_res['messages'][-1].pretty_print()

================================== Ai Message ==================================

It seems like there might be a bit of confusion. It looks like you're identifying as "Joey Tribbiani," which is a character from the TV show "Friends." But in this conversation, we are talking about me, Qwen.

To clarify who you are: You seem to have identified yourself as Joey Tribbiani, but in our current context, I am Qwen. How can I assist you further with information related to "Friends" or any other topic?


In [ ]:
from langchain.tools import tool, ToolRuntime
# class CustomState(AgentState):
#     user_id:str

# @tool
# def get_user_info(runtime:ToolRuntime) -> str:
#     """Look up user information."""
#     user_id = runtime.state['user_id']
#     return "User is Ross" if user_id=='user123' else "Unknown User"

# agent = create_agent(
#     model=model,
#     tools=[get_user_info],
#     state_schema=CustomState
# )
# result = agent.invoke({'messages':'Find user info.', 'user_id':'user123'})
# result

{'messages': [HumanMessage(content='Find user info.', additional_kwargs={}, response_metadata={}, id='7f96cfcc-c50e-4f0b-98d5-c6be7b75de9a'),
  AIMessage(content="Sure, I can help with that. Could you please provide me with the ID or any other identifier of the user whose information you need to find? Without this, I won't be able to proceed. If you're not sure how to get this information, let me know and I'll assist further.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-25T11:25:05.1266226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21042514100, 'load_duration': 3208239900, 'prompt_eval_count': 139, 'prompt_eval_duration': 8473427400, 'eval_count': 62, 'eval_duration': 9235144300, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bf4e6-2cae-7e72-8b3b-de46433b160a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 139, 'output_tokens': 62, 'total_tokens': 201})],
 'user_id': 'user123'}

In [25]:
from pydantic import BaseModel
from langgraph.types import Command
from langchain.messages import ToolMessage

class CustomState(AgentState):
    user_name:str
class CustomContext(BaseModel):
    user_id:str

@tool
def update_user_info(runtime:ToolRuntime[CustomContext, CustomState]) -> Command:
    """Look up and update user info."""
    user_id = runtime.context.user_id
    name = "Ross" if user_id=='user123' else "Unknown User"
    return Command(
        update={
            'user_name':name,
            'messages':[
                ToolMessage("Successfully found user info.",
                tool_call_id = runtime.tool_call_id),
            ]
        }
    )

@tool
def greet_user(runtime:ToolRuntime[CustomContext, CustomState]) -> str|Command:
    """Use this to greet user once you found their info."""
    user_name = runtime.state.get('user_name', None)
    if user_name is None:
        return Command(
            update={
                'messages':[
                    ToolMessage("Please call `update_user_info` tool, it will get and update the user info.",
                                tool_call_id = runtime.tool_call_id)
                ]
            }
        )
    return f"Hello {user_name}!"
agent = create_agent(
    model=model,
    # system_prompt="Use tools to get and update user info and greet the user once",
    tools=[update_user_info, greet_user],
    context_schema=CustomContext,
    state_schema=CustomState
)
result = agent.invoke(
    {"messages":[{'role':'user','content':'greet the user'}]},
    context=CustomContext(user_id='user123')
)
result

{'messages': [HumanMessage(content='greet the user', additional_kwargs={}, response_metadata={}, id='cc74951c-e56e-426c-a868-9dd5de2552ac'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-25T13:06:11.0505168Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9194410100, 'load_duration': 115333000, 'prompt_eval_count': 178, 'prompt_eval_duration': 7362193300, 'eval_count': 17, 'eval_duration': 1692460600, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019bf542-e9fd-7e91-80e6-2a62d169c1cb-0', tool_calls=[{'name': 'greet_user', 'args': {}, 'id': 'bcbc1e59-0c70-4cdc-9942-9c9f05759a31', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 17, 'total_tokens': 195}),
  ToolMessage(content='Please call `update_user_info` tool, it will get and update the user info.', name='greet_user', id='cae4a64a-d8ab-479a-b392-be42603c4a24', tool_call_id='bcbc1e59-0c70-4

In [33]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest
class CustomContext(TypedDict):
    user_name:str

@tool 
def get_weather(city:str) -> str:
    """Get weather forecast for a city."""
    return f"The weather in {city} is always sunny."

@dynamic_prompt
def dynamic_system_prompt(request:ModelRequest)->str:
    """Provide a dynamic system prompt to the assistant."""
    # user_name = request.state.get(user_name)
    user_name = request.runtime.context['user_name']
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}"
    return system_prompt

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[dynamic_system_prompt],
    context_schema=CustomContext
)
response = agent.invoke(
    {'messages':[{'role':'user', 'content':"What's the weather in Pune, Maharastra?"}]},
    context=CustomContext(user_name='Sheldon')
)
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

What's the weather in Pune, Maharastra?
================================== Ai Message ==================================
Tool Calls:
  get_weather (cce565d6-c7f0-4a6c-9fd8-7127b2d58ece)
 Call ID: cce565d6-c7f0-4a6c-9fd8-7127b2d58ece
  Args:
    city: Pune
================================= Tool Message =================================
Name: get_weather

The weather in Pune is always sunny.
================================== Ai Message ==================================

The weather forecast for Pune, Maharashtra indicates that it will be sunny.
